In [24]:
import os

# Specify the directory you want to create
new_directory = "/kaggle/working/images"

# Create the directory (this will not throw an error if the directory already exists)
os.makedirs(new_directory, exist_ok=True)

print(f"Directory created at: {new_directory}")


In [25]:
import shutil

# Define the source and destination directories
source_directory = "/kaggle/input/annotated-images-bhf/"  # Update with your actual path
destination_directory = "/kaggle/working/images/"  # The new directory you created

# Move all images from source to destination
for filename in os.listdir(source_directory):
    source_path = os.path.join(source_directory, filename)
    destination_path = os.path.join(destination_directory, filename)

    # Move the file
    shutil.copy(source_path, destination_path)

print(f"Moved files to: {destination_directory}")


In [26]:
import torch
import torchvision
import torchvision.transforms as T
from torch import optim
import os
import numpy as np
import json
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader

In [27]:

class ECGDataset(Dataset):
    def __init__(self, img_dir, annotation_file, transforms=None):
        self.img_dir = img_dir
        self.transforms = transforms

        # Load COCO annotations
        with open(annotation_file) as f:
            self.coco_data = json.load(f)

        self.image_ids = [img["id"] for img in self.coco_data["images"]]

        # Create a mapping from image_id to annotations
        self.ann_dict = {img_id: [] for img_id in self.image_ids}
        for ann in self.coco_data["annotations"]:
            self.ann_dict[ann["image_id"]].append(ann)

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        # Get image ID
        img_id = self.image_ids[idx]
        img_info = next(img for img in self.coco_data["images"] if img["id"] == img_id)
        img_path = os.path.join(self.img_dir, img_info["file_name"])
        
        # Load image
        image = Image.open(img_path).convert("RGB")
        w, h = image.size

        # Load annotations (bounding boxes)
        boxes = []
        labels = []
        for ann in self.ann_dict[img_id]:
            x, y, width, height = ann["bbox"]
            boxes.append([x, y, x + width, y + height])
            labels.append(1)  # "1" is for ECG class

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        # Create target dictionary
        target = {
            "boxes": boxes,
            "labels": labels
        }

        if self.transforms:
            image = self.transforms(image)

        return image, target


In [28]:
annotation_file = '/kaggle/input/annotated-images-reference/result.json'

In [29]:
img_dir = '/kaggle/working/'

In [30]:
transform = T.Compose([T.ToTensor()])
train_dataset = ECGDataset(img_dir, annotation_file, transforms=transform)
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, collate_fn=lambda batch: tuple(zip(*batch)))


In [31]:
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# Load pretrained Faster R-CNN model
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights='DEFAULT')

# Modify classifier (for 1 class + background)
num_classes = 2  # 1 ECG class + background
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

In [32]:
# Move model to GPU if available
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu"))
model.to(device)
print(f"Device: {device}")

Device: cuda


In [33]:
# Define optimizer
optimizer = optim.Adam(model.parameters(), lr=0.0001)

In [34]:
from tqdm import tqdm

# Train the model for 10 epochs
num_epochs = 10

for epoch in range(num_epochs):
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=True)

    # Track total loss
    total_loss = 0.0
    model.train()

    for images, targets in progress_bar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Update tqdm progress bar with current loss
        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Avg Loss: {avg_loss:.4f}")


Epoch 1/10: 100%|██████████| 25/25 [00:54<00:00,  2.18s/it, loss=0.0521]


Epoch [1/10], Avg Loss: 0.1681


Epoch 2/10: 100%|██████████| 25/25 [00:54<00:00,  2.17s/it, loss=0.0491]


Epoch [2/10], Avg Loss: 0.0621


Epoch 3/10: 100%|██████████| 25/25 [00:54<00:00,  2.16s/it, loss=0.0337]


...

Epoch [8/10], Avg Loss: 0.0337


Epoch 9/10: 100%|██████████| 25/25 [00:54<00:00,  2.17s/it, loss=0.0198]


Epoch [9/10], Avg Loss: 0.0276


Epoch 10/10: 100%|██████████| 25/25 [00:54<00:00,  2.17s/it, loss=0.0185]

Epoch [10/10], Avg Loss: 0.0246


In [49]:
# Save the entire model
torch.save(model.state_dict(), "trained_ecg_object_detection_model.pth")